# TPC-H Data Transformation Pipeline

## Overview

This notebook performs data transformation on the TPC-H sample dataset. It reads and joins three core tables:

* **samples.tpch.orders** - Order transactions (7.5M records, 1992-1998)
* **samples.tpch.customer** - Customer master data (750K records)
* **samples.tpch.nation** - Nation reference data (25 records)

The notebook creates an enriched view `customer_orders_enriched` that combines order details with customer information and nation names, providing a foundation for business intelligence analysis.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_orders_enriched AS
SELECT 
  o.o_orderkey,
  o.o_orderdate,
  o.o_totalprice,
  o.o_orderstatus,
  o.o_orderpriority,
  c.c_custkey,
  c.c_name,
  c.c_acctbal,
  c.c_mktsegment,
  n.n_name AS nation_name,
  n.n_regionkey
FROM samples.tpch.orders o
INNER JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
INNER JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey;

In [0]:
%sql
-- Data Quality Summary
SELECT 
  'Total Records' AS metric,
  COUNT(*) AS value,
  NULL AS details
FROM customer_orders_enriched

UNION ALL

SELECT 
  'Date Range' AS metric,
  NULL AS value,
  CONCAT(MIN(o_orderdate), ' to ', MAX(o_orderdate)) AS details
FROM customer_orders_enriched

UNION ALL

SELECT 
  'Null Order Keys' AS metric,
  SUM(CASE WHEN o_orderkey IS NULL THEN 1 ELSE 0 END) AS value,
  NULL AS details
FROM customer_orders_enriched

UNION ALL

SELECT 
  'Null Customer Keys' AS metric,
  SUM(CASE WHEN c_custkey IS NULL THEN 1 ELSE 0 END) AS value,
  NULL AS details
FROM customer_orders_enriched

UNION ALL

SELECT 
  'Null Nation Names' AS metric,
  SUM(CASE WHEN nation_name IS NULL THEN 1 ELSE 0 END) AS value,
  NULL AS details
FROM customer_orders_enriched

UNION ALL

SELECT 
  'Distinct Nations' AS metric,
  COUNT(DISTINCT nation_name) AS value,
  NULL AS details
FROM customer_orders_enriched

UNION ALL

SELECT 
  'Distinct Customers' AS metric,
  COUNT(DISTINCT c_custkey) AS value,
  NULL AS details
FROM customer_orders_enriched;

metric,value,details
Total Records,7500000,null
Date Range,null,1992-01-01 to 1998-08-02
Null Order Keys,0,null
Null Customer Keys,0,null
Null Nation Names,0,null
Distinct Nations,25,null
Distinct Customers,499989,null


In [0]:
%sql
-- Sample of enriched data
SELECT 
  o_orderkey,
  o_orderdate,
  o_totalprice,
  o_orderstatus,
  c_name,
  nation_name,
  c_mktsegment
FROM customer_orders_enriched
ORDER BY o_orderdate DESC
LIMIT 10;

o_orderkey,o_orderdate,o_totalprice,o_orderstatus,c_name,nation_name,c_mktsegment
12639175,1998-08-02,49293.14,O,Customer#000643105,INDIA,BUILDING
10646919,1998-08-02,289275.43,O,Customer#000402367,JAPAN,MACHINERY
10896100,1998-08-02,380501.17,O,Customer#000461533,SAUDI ARABIA,BUILDING
11614594,1998-08-02,105988.68,O,Customer#000097795,ARGENTINA,MACHINERY
3152581,1998-08-02,55129.09,O,Customer#000308636,EGYPT,FURNITURE
2889445,1998-08-02,265921.80,O,Customer#000421750,IRAN,HOUSEHOLD
19822693,1998-08-02,104325.72,O,Customer#000105478,RUSSIA,BUILDING
6704035,1998-08-02,182602.33,O,Customer#000545446,CHINA,BUILDING
5374759,1998-08-02,171089.63,O,Customer#000142442,ROMANIA,HOUSEHOLD
24815424,1998-08-02,54381.86,O,Customer#000064057,JAPAN,FURNITURE
